# Lead Ranker — Transformer Fine-Tuning, Dataset Hashing, and Model Card

This notebook fine-tunes and compares two Hugging Face transformer models for the lead-ranker task.

Task:

- `0` = normal lead
- `1` = hot lead

Expected input file:

- `lead_ranker_hot_normal_data.csv`

Expected columns:

- `text`
- `label`

Models compared:

1. `microsoft/deberta-v3-small`
2. `answerdotai/ModernBERT-base`

The notebook includes:

- train / validation / test split
- dataset SHA-256 hashing
- Hugging Face model loading
- fine-tuning with `Trainer`
- evaluation metrics
- confusion matrices
- best model selection
- best model export
- production-style model card exported as Markdown and JSON

## 1. Colab setup

Use a GPU runtime if possible:

`Runtime > Change runtime type > T4 GPU`

In [ ]:
!pip -q install -U "transformers>=4.48.0" datasets accelerate evaluate sentencepiece scikit-learn pandas numpy matplotlib safetensors

## 2. Optional: upload the dataset

Run this cell if `lead_ranker_hot_normal_data.csv` is not already visible in the Colab file browser.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except ImportError:
    print("This upload helper only works inside Google Colab.")

## 3. Imports and configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import inspect
import json
import os
import platform
import random
import shutil
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

DATA_PATH = "lead_ranker_hot_normal_data.csv"

OUTPUT_ROOT = Path("lead_ranker_transformer_runs")
BEST_MODEL_DIR = Path("lead_ranker_best_transformer_model")
BEST_MODEL_ZIP = Path("lead_ranker_best_transformer_model.zip")

MODEL_CARD_MD_PATH = Path("lead_ranker_model_card.md")
MODEL_CARD_JSON_PATH = Path("lead_ranker_model_card.json")
SUMMARY_JSON_PATH = Path("lead_ranker_transformer_comparison_summary.json")

RANDOM_STATE = 42
MAX_LENGTH = 128
NUM_LABELS = 2

NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

LABEL_NAMES = {
    0: "normal",
    1: "hot",
}

ID2LABEL = {
    0: "normal",
    1: "hot",
}

LABEL2ID = {
    "normal": 0,
    "hot": 1,
}

MODEL_CANDIDATES = {
    "deberta_v3_small": "microsoft/deberta-v3-small",
    "modernbert_base": "answerdotai/ModernBERT-base",
}

OUTPUT_ROOT.mkdir(exist_ok=True)

set_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Load, validate, and fingerprint the dataset

The notebook creates two hashes:

- **Raw file SHA-256**: hash of the exact CSV bytes.
- **Loaded content SHA-256**: hash of the cleaned dataframe after loading and validation.

These hashes are stored in the model card and comparison summary for reproducibility.

In [ ]:
def sha256_file(path):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_dataframe_content(dataframe):
    canonical_csv = dataframe.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(canonical_csv.encode("utf-8")).hexdigest()


if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Upload it to the Colab workspace first."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"text", "label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df[["text", "label"]].copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df = df[df["text"] != ""].copy()
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

dataset_metadata = {
    "dataset_path": DATA_PATH,
    "dataset_file_size_bytes": Path(DATA_PATH).stat().st_size,
    "dataset_raw_sha256": sha256_file(DATA_PATH),
    "dataset_content_sha256": sha256_dataframe_content(df),
    "rows": int(len(df)),
    "columns": list(df.columns),
    "label_distribution": {
        str(k): int(v)
        for k, v in df["label"].value_counts().sort_index().items()
    },
    "duplicate_text_rows": int(df.duplicated(subset=["text"]).sum()),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(dataset_metadata, indent=2))
display(df.head(10))

## 5. Train / validation / test split

Split:

- 70% train
- 15% validation
- 15% test

The split is stratified.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

split_metadata = {
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "train_label_distribution": {
        str(k): int(v)
        for k, v in train_df["label"].value_counts().sort_index().items()
    },
    "validation_label_distribution": {
        str(k): int(v)
        for k, v in val_df["label"].value_counts().sort_index().items()
    },
    "test_label_distribution": {
        str(k): int(v)
        for k, v in test_df["label"].value_counts().sort_index().items()
    },
}

print(json.dumps(split_metadata, indent=2))

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
    }
)

dataset

## 6. Metrics and training helpers

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0,
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def tokenize_dataset(raw_dataset, tokenizer):
    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized = raw_dataset.map(tokenize_batch, batched=True)
    tokenized = tokenized.remove_columns(["text"])
    tokenized.set_format("torch")
    return tokenized


def make_training_arguments(output_dir):
    kwargs = {
        "output_dir": str(output_dir),
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "num_train_epochs": NUM_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "save_strategy": "epoch",
        "logging_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "save_total_limit": 1,
        "report_to": "none",
        "seed": RANDOM_STATE,
        "fp16": torch.cuda.is_available(),
    }

    signature = inspect.signature(TrainingArguments.__init__)

    if "eval_strategy" in signature.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    return TrainingArguments(**kwargs)


def load_sequence_classifier(model_name):
    model_kwargs = {
        "num_labels": NUM_LABELS,
        "id2label": ID2LABEL,
        "label2id": LABEL2ID,
    }

    try:
        return AutoModelForSequenceClassification.from_pretrained(
            model_name,
            attn_implementation="eager",
            **model_kwargs,
        )
    except TypeError:
        return AutoModelForSequenceClassification.from_pretrained(
            model_name,
            **model_kwargs,
        )

## 7. Fine-tune and evaluate one model

In [ ]:
def train_and_evaluate_model(model_key, model_name):
    print("=" * 90)
    print(f"Fine-tuning: {model_key} -> {model_name}")
    print("=" * 90)

    run_dir = OUTPUT_ROOT / model_key
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    tokenized_dataset = tokenize_dataset(dataset, tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = load_sequence_classifier(model_name)
    training_args = make_training_arguments(run_dir)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    val_metrics = trainer.evaluate(tokenized_dataset["validation"])
    test_output = trainer.predict(tokenized_dataset["test"])

    test_logits = test_output.predictions
    test_labels = test_output.label_ids
    test_predictions = np.argmax(test_logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels,
        test_predictions,
        average="binary",
        zero_division=0,
    )

    accuracy = accuracy_score(test_labels, test_predictions)

    print("\nTest classification report:")
    print(
        classification_report(
            test_labels,
            test_predictions,
            target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
            zero_division=0,
        )
    )

    cm = confusion_matrix(test_labels, test_predictions)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[LABEL_NAMES[0], LABEL_NAMES[1]],
    )
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix - {model_key}")
    plt.show()

    saved_model_dir = run_dir / "final_model"
    trainer.save_model(saved_model_dir)
    tokenizer.save_pretrained(saved_model_dir)

    result = {
        "model_key": model_key,
        "model_name": model_name,
        "saved_model_dir": str(saved_model_dir),
        "validation_accuracy": float(val_metrics.get("eval_accuracy", np.nan)),
        "validation_precision": float(val_metrics.get("eval_precision", np.nan)),
        "validation_recall": float(val_metrics.get("eval_recall", np.nan)),
        "validation_f1": float(val_metrics.get("eval_f1", np.nan)),
        "test_accuracy": float(accuracy),
        "test_precision": float(precision),
        "test_recall": float(recall),
        "test_f1": float(f1),
        "confusion_matrix": cm.tolist(),
    }

    del model
    del trainer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## 8. Fine-tune and compare both models

This may take several minutes per model on a Colab T4 GPU.

In [ ]:
results = []

for model_key, model_name in MODEL_CANDIDATES.items():
    result = train_and_evaluate_model(model_key, model_name)
    results.append(result)

results_df = pd.DataFrame(results).sort_values(
    by="test_f1",
    ascending=False,
).reset_index(drop=True)

display(results_df)

## 9. Select and export the best model

In [ ]:
best_row = results_df.iloc[0].to_dict()
best_model_key = best_row["model_key"]
best_model_name = best_row["model_name"]
best_saved_model_dir = Path(best_row["saved_model_dir"])

print("Best model key:", best_model_key)
print("Best Hugging Face model:", best_model_name)
print("Best model directory:", best_saved_model_dir)

if BEST_MODEL_DIR.exists():
    shutil.rmtree(BEST_MODEL_DIR)

shutil.copytree(best_saved_model_dir, BEST_MODEL_DIR)

summary = {
    "task": "lead_ranker_hot_vs_normal",
    "label_mapping": LABEL_NAMES,
    "dataset_metadata": dataset_metadata,
    "split_metadata": split_metadata,
    "training_config": {
        "random_state": RANDOM_STATE,
        "max_length": MAX_LENGTH,
        "num_epochs": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "selection_metric": "test_f1",
    },
    "models_compared": MODEL_CANDIDATES,
    "results": results_df.to_dict(orient="records"),
    "best_model": best_row,
    "best_model_export_dir": str(BEST_MODEL_DIR),
    "runtime": {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    },
}

SUMMARY_JSON_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Copied best model to: {BEST_MODEL_DIR}")
print(f"Saved comparison summary to: {SUMMARY_JSON_PATH}")

## 10. Generate production-style model card

The model card is exported as both:

- `lead_ranker_model_card.md`
- `lead_ranker_model_card.json`

In [ ]:
def metric(value):
    if isinstance(value, (float, np.floating)):
        return f"{value:.4f}"
    return str(value)


model_card = {
    "model_card_version": "1.0",
    "model_name": "lead_ranker_hot_normal",
    "model_type": "transformer_sequence_classifier",
    "task": "binary_text_classification",
    "selected_model_key": best_model_key,
    "selected_hugging_face_model": best_model_name,
    "exported_model_dir": str(BEST_MODEL_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "labels": {
        "0": "normal",
        "1": "hot",
    },
    "intended_use": {
        "primary_use": "Rank accepted real-estate leads as normal or hot.",
        "upstream_dependency": "This model assumes the message has already passed the Stage 1 lead gatekeeper.",
        "recommended_use": "Use the hot probability to prioritize follow-up, routing, or notification urgency.",
    },
    "out_of_scope_use": [
        "Do not use as a spam detector.",
        "Do not use as a final sales qualification decision system.",
        "Do not use for legal, financial, credit, or housing eligibility decisions.",
        "Do not use on production traffic without monitoring and periodic review.",
    ],
    "dataset": dataset_metadata,
    "splits": split_metadata,
    "training_config": summary["training_config"],
    "models_compared": MODEL_CANDIDATES,
    "evaluation": {
        "selection_metric": "test_f1",
        "all_results": results_df.to_dict(orient="records"),
        "selected_model_metrics": best_row,
    },
    "limitations": [
        "The dataset is synthetic, so real app traffic may contain phrasing not represented here.",
        "The model may learn template-specific patterns.",
        "Hot-vs-normal ranking is subjective and should be reviewed against business outcomes.",
        "A hot prediction should prioritize follow-up, not guarantee deal quality.",
        "The model was not validated on multilingual production messages unless such messages are added to the dataset.",
    ],
    "monitoring_recommendations": [
        "Log prediction, hot probability, message length, timestamp, and downstream outcome.",
        "Review false hot predictions that waste agent time.",
        "Review false normal predictions that later became high-value leads.",
        "Track class distribution drift over time.",
        "Retrain with real reviewed leads once available.",
    ],
    "retraining_recommendations": [
        "Add real production messages with reviewed hot/normal labels.",
        "Add borderline cases such as 'price please' vs 'ready to pay deposit today'.",
        "Include multilingual and shorthand user messages if expected in the app.",
        "Keep a held-out real-world evaluation set separate from synthetic training data.",
    ],
    "runtime": summary["runtime"],
}


model_card_md = f"""# Model Card: Lead Ranker Hot vs Normal

## Model Details

- **Model name:** `lead_ranker_hot_normal`
- **Model type:** Transformer sequence classifier
- **Selected model:** `{best_model_name}`
- **Selected model key:** `{best_model_key}`
- **Exported model directory:** `{BEST_MODEL_DIR}`
- **Created at UTC:** `{model_card["created_at_utc"]}`

## Intended Use

This model ranks real-estate leads as either:

- `0` = normal
- `1` = hot

It should be used after the Stage 1 gatekeeper has already decided that a message is a lead.

Recommended use: prioritize agent follow-up, routing, alerts, and CRM urgency.

## Out-of-Scope Use

- Not a spam detector.
- Not a final sales qualification system.
- Not for legal, financial, credit, or housing eligibility decisions.
- Not validated for production traffic without monitoring and review.

## Dataset Fingerprint

- **Dataset path:** `{dataset_metadata["dataset_path"]}`
- **Rows:** `{dataset_metadata["rows"]}`
- **Columns:** `{dataset_metadata["columns"]}`
- **Label distribution:** `{dataset_metadata["label_distribution"]}`
- **Raw file SHA-256:** `{dataset_metadata["dataset_raw_sha256"]}`
- **Loaded content SHA-256:** `{dataset_metadata["dataset_content_sha256"]}`
- **File size bytes:** `{dataset_metadata["dataset_file_size_bytes"]}`
- **Duplicate text rows:** `{dataset_metadata["duplicate_text_rows"]}`

## Data Splits

- **Train rows:** `{split_metadata["train_rows"]}`
- **Validation rows:** `{split_metadata["validation_rows"]}`
- **Test rows:** `{split_metadata["test_rows"]}`
- **Train label distribution:** `{split_metadata["train_label_distribution"]}`
- **Validation label distribution:** `{split_metadata["validation_label_distribution"]}`
- **Test label distribution:** `{split_metadata["test_label_distribution"]}`

## Training Configuration

- **Max sequence length:** `{MAX_LENGTH}`
- **Epochs:** `{NUM_EPOCHS}`
- **Train batch size:** `{TRAIN_BATCH_SIZE}`
- **Eval batch size:** `{EVAL_BATCH_SIZE}`
- **Gradient accumulation steps:** `{GRADIENT_ACCUMULATION_STEPS}`
- **Learning rate:** `{LEARNING_RATE}`
- **Weight decay:** `{WEIGHT_DECAY}`
- **Random state:** `{RANDOM_STATE}`
- **Selection metric:** `test_f1`

## Models Compared

{pd.DataFrame([{"model_key": k, "hugging_face_model": v} for k, v in MODEL_CANDIDATES.items()]).to_markdown(index=False)}

## Evaluation Results

{results_df.drop(columns=["confusion_matrix"], errors="ignore").to_markdown(index=False)}

## Selected Model Metrics

- **Validation F1:** {metric(best_row["validation_f1"])}
- **Test Accuracy:** {metric(best_row["test_accuracy"])}
- **Test Precision:** {metric(best_row["test_precision"])}
- **Test Recall:** {metric(best_row["test_recall"])}
- **Test F1:** {metric(best_row["test_f1"])}

## Limitations

- The dataset is synthetic, so real app traffic may contain phrasing not represented here.
- The model may learn template-specific patterns.
- Hot-vs-normal ranking is subjective and should be reviewed against business outcomes.
- A hot prediction should prioritize follow-up, not guarantee deal quality.
- The model was not validated on multilingual production messages unless such messages are added to the dataset.

## Monitoring Recommendations

- Log prediction, hot probability, message length, timestamp, and downstream outcome.
- Review false hot predictions that waste agent time.
- Review false normal predictions that later became high-value leads.
- Track class distribution drift over time.
- Retrain with real reviewed leads once available.

## Retraining Recommendations

- Add real production messages with reviewed hot/normal labels.
- Add borderline cases such as `price please` vs `ready to pay deposit today`.
- Include multilingual and shorthand user messages if expected in the app.
- Keep a held-out real-world evaluation set separate from synthetic training data.

## Runtime

- **Python:** `{sys.version.split()[0]}`
- **Platform:** `{platform.platform()}`
- **PyTorch:** `{torch.__version__}`
- **CUDA available:** `{torch.cuda.is_available()}`
"""


MODEL_CARD_MD_PATH.write_text(model_card_md, encoding="utf-8")
MODEL_CARD_JSON_PATH.write_text(json.dumps(model_card, indent=2), encoding="utf-8")

print(f"Saved Markdown model card to: {MODEL_CARD_MD_PATH}")
print(f"Saved JSON model card to: {MODEL_CARD_JSON_PATH}")

## 11. Preview the model card

In [ ]:
print(MODEL_CARD_MD_PATH.read_text(encoding="utf-8"))

## 12. Zip best model and artifacts

In [ ]:
if BEST_MODEL_ZIP.exists():
    BEST_MODEL_ZIP.unlink()

zip_path = shutil.make_archive(
    base_name=str(BEST_MODEL_DIR),
    format="zip",
    root_dir=BEST_MODEL_DIR,
)

print(f"Zipped best model to: {zip_path}")
print(f"Summary JSON: {SUMMARY_JSON_PATH}")
print(f"Model card MD: {MODEL_CARD_MD_PATH}")
print(f"Model card JSON: {MODEL_CARD_JSON_PATH}")

## 13. Production-style inference test

In [ ]:
best_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR, use_fast=True)
best_model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)
best_model.eval()


def predict_lead_rank(texts):
    inputs = best_tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = best_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    predictions = probabilities.argmax(axis=1)

    rows = []
    for text, pred, probs in zip(texts, predictions, probabilities):
        rows.append(
            {
                "text": text,
                "prediction": int(pred),
                "prediction_name": LABEL_NAMES[int(pred)],
                "normal_probability": float(probs[0]),
                "hot_probability": float(probs[1]),
            }
        )

    return pd.DataFrame(rows)


examples = [
    "Price please",
    "Is this apartment still available?",
    "I have $1,200/month and can view the apartment in Hamra tomorrow",
    "I can pay the deposit today if the apartment is available",
    "Can you send pictures?",
    "I own an apartment in Achrafieh and need you to list it ASAP",
]

display(predict_lead_rank(examples))

## 14. Optional: download model, model card, and summary from Colab

In [ ]:
try:
    from google.colab import files

    files.download(str(BEST_MODEL_ZIP))
    files.download(str(MODEL_CARD_MD_PATH))
    files.download(str(MODEL_CARD_JSON_PATH))
    files.download(str(SUMMARY_JSON_PATH))
except ImportError:
    print("Download helper only works inside Google Colab.")